# Simple Linear Regression - An Example

Suppose that a sports analyst wants to know if there exists a relation between the number of times baseball player step at the plate and the number of runs batted in. In case that a relationship exists, we can use an appropriate model for predicting the outcome of a game.

## Libraries

These are the libraries used in this analysis, together with some setup options:

In [ ]:
# Data wrangling
# ==============================================================================
import pandas as pd
import numpy as np

In [ ]:
# Plots
# ==============================================================================
import matplotlib.pyplot as plt
from matplotlib import style
import seaborn as sns

In [ ]:
# Preprocessing and modeling
# ==============================================================================
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

In [ ]:
# Configuring matplotlib and seaborn
# ==========================================================================bb====
from matplotlib.colors import ListedColormap
plt.rcParams['savefig.bbox'] = "tight"
sns.set_style('whitegrid') # or plt.style.use('ggplot')
sns.set_palette('rainbow')

In [ ]:
# For a list of colormaps uncomment the following lines
#from matplotlib import colormaps
#sorted(list(colormaps))

In [ ]:
# Configuring warnings
# ==============================================================================
# If you want to suppress warnings uncomment the following lines
#import warnings
#warnings.filterwarnings('ignore')

## Data

In [ ]:
# Data
# ==============================================================================
teams = ["Texas","Boston","Detroit","Kansas","St.","New_S.","New_Y.",
           "Milwaukee","Colorado","Houston","Baltimore","Los_An.","Chicago",
           "Cincinnati","Los_P.","Philadelphia","Chicago","Cleveland","Arizona",
           "Toronto","Minnesota","Florida","Pittsburgh","Oakland","Tampa",
           "Atlanta","Washington","San.F","San.I","Seattle"]
at_bat = [5659,  5710, 5563, 5672, 5532, 5600, 5518, 5447, 5544, 5598,
          5585, 5436, 5549, 5612, 5513, 5579, 5502, 5509, 5421, 5559,
          5487, 5508, 5421, 5452, 5436, 5528, 5441, 5486, 5417, 5421]

runs = [855, 875, 787, 730, 762, 718, 867, 721, 735, 615, 708, 644, 654, 735,
        667, 713, 654, 704, 731, 743, 619, 625, 610, 645, 707, 641, 624, 570,
        593, 556]

data = pd.DataFrame({'teams': teams, 'at_bat': at_bat, 'runs': runs})
mean_predicted_variable = data['runs'].mean()
data.head()

In [ ]:
data.describe()

## Graphical analysis

First of all, we plot the data in order to see if there is a relation between the variables and calculate correlation coefficients.

In [ ]:
# Scatter plot
# ==============================================================================
fig, ax = plt.subplots(figsize=(6, 3.84))

sns.scatterplot(
    data = data,
    x    = 'at_bat',
    y    = 'runs',
    ax   = ax
)

ax.set_title('Runs vs At-bat');

In [ ]:
# Linear correlation
# ==============================================================================
corr_test = pearsonr(x = data['at_bat'], y =  data['runs'])
print("Pearson correlation coefficient: ", corr_test[0])
print("P-value: ", corr_test[1])

The graph and the correlation test show a linear relation, with a relatively high intensity (r = 0.61) and high significance level (p-value = 0.000339). Thus, we can proceed to generate a linear regression model so we can predict the number of runs as a function of the number of At-bats of a team.

## Fitting the model

Let's fit a model with `runs` as the response variable, and `at_bat` as the predictor. In order to do this, we split the data in two groups, a training set and a test set.

As usual, there are several methods we can use for fitting a linear regression model with Python. We can use either **Scikit-learn** or **Statsmodels**. Both libraries give pretty much the same model, however **Statsmodels** has more options and we can do a more comprehensive analysis with it.

In either case, we begin by splitting the dataframe into two: a training set which is used for fitting the model, and a test set which is used to validate it. After we have fitted and validated the model, we evaluate its predicting strength by calculating the *root mean squared error (`rmse`)*, which is a measure of a distance between the real data and the predictions. As a rule of thumb the `rmse` must be no greater than 10% of the mean of the predicted variable.

### Scikit-learn

With **Scikit-learn** we basically just fit the model to the training set and validate it with the test set, using the root-square mean error and comparing it with the mean of `runs`.

In [ ]:
# Train-test Splitting
# ==============================================================================
X = data[['at_bat']]
y = data['runs']

X_train, X_test, y_train, y_test = train_test_split(
                                        X.values.reshape(-1,1),
                                        y.values.reshape(-1,1),
                                        train_size   = 0.8,
                                        random_state = 1234,
                                        shuffle      = True
                                    )

In [ ]:
# Model fitting
# ==============================================================================
model = LinearRegression()
model.fit(X = X_train.reshape(-1, 1), y = y_train)

In [ ]:
# Model summary
# ==============================================================================
print("Intercept:", model.intercept_)
print("Coefficient:", list(zip(X.columns, model.coef_.flatten(), )))
print("R^2 determination coefficient:", model.score(X.values, y))

Once we trained the model, we evaluate its predictive strength using the test set.

In [ ]:
# Model error test 
# ==============================================================================
predictions = model.predict(X = X_test)
print(predictions[0:3,])

rmse = root_mean_squared_error(
        y_true  = y_test,
        y_pred  = predictions
       )
print("")
print(f"The test error (rmse) is {rmse}, which is {rmse/mean_predicted_variable*100:.2f}% of the mean of `runs`.")

### Statsmodels

The linear regression implementation in **Statsmodels** is more complete than that of **Scikit-learn** since, in addition to fitting the model, we can estimate the necessary test statistics and analyses for verifying that model conditions are satisfied. **Statsmodels** provides two ways of training the model:

- Giving a formula for the model and feeding it with a `DataFrame` for the training test that includes the response variable and the predictors. This is similar to the method used in **R**.

- Giving two matrices, a predictor matrix and a response matrix. This is similar to the method used with **Scikit-learn**, only that the predictor matrix needs to have all 1s as the first column.

The outcome of our analisys is a table that displays all the important information gathered in the Linear Regression. Here follows an explanation of each item in the table.

- **Dep variable:** The dependent variable in the data.
- **Model and Method:** OLS which stands for Ordinary Least Square. The model tries to find out a linear expression for the dataset which minimizes the sum of residual squares.
- **DF residuals and DF model:** `DF Model` is the number of independent variables. `DF residual` is the difference between the number of observations and the number of features.
- **Covariance type:** Covariance type is typically nonrobust which means there is no elimination of data to calculate the covariance between features. `Covariance` shows how two variables move with respect to each other. If this value is greater than 0, both move in same direction and if it is less than 0, the variables move in opposite directions. Covariance is difference from correlation. Covariance does not provide the strength of the relationship, only the direction of movement whereas the correlation value is normalized and ranges between -1 to +1 and provides the strength of relationship. If we want to obtain robust covariance, we can declare `cov_type=HC0/HC1/HC2/HC3`. However, the `statsmodel` documentation is not that rich to explain all these. HC stands for heteroscedasticity-consistent and HC0 implements the simplest version among all.
- **R-squared:** The R-squared value is the coefficient of determination which indicates the percentage of the variability in the data explained by the selected independent variables.
- **Adj. R-squared:** As we add more and more independent variables to our model, the R-squared value increases but in reality, those variables do not necessarily make any contribution towards explaining the dependent variable. Therefore addition of each unnecessary variables needs some sort of penalty. The original R-squared value is adjusted when there are multiple variables incorporated. In essence, we should always look for adjusted R-squared value while performing multiple linear regression. For a single independent variable, both R-squared and adjusted R-squared value are same.
- **coef** and **std err:** The `coef` column represents the coefficients for each independent variable along with the intercept value. `Std err` is the standard deviation of the corresponding variable's coefficient across all the data points.
- **$t$-values and $P>|t|$:** The $t$-column provides the $t$-values corresponding to each independent variable. $t$-statistics are used to calculate the $p$-values. Typically when $p$-value is less than $0.05$ it indicates a strong evidence against the null hypothesis which states that the corresponding independent variable has no effect on the dependent variable.
- **$F$-statistics:** An $F$-test provides a way to check all the independent variables together to see if any of those are related to the dependent variable. If `Prob(F-statistic)` is greater than $0.05$ there is no evidence of relationship between any of the independent variables with the output. If it is less than $0.05$, we can say that there is at least one variable which is significantly related with the output. However, there may be some cases when `prob(F-statistic)` may be greater than $0.05$ but one of the independent variables shows strong correlation. This is because each $t$-test is carried out with different sets of data whereas the $F$-test checks the combined effect including all variables globally.
- **Log-likelihood:** The `log-likelihood` value is a measure of how good the model fits the given data. It is useful when we compare two or more models. The higher the value of `log-likelihood`, the better the model fits the given data. It can range from negative infinity to positive infinity.
- **AIC and BIC:** AIC, which stands for Akaike’s Information Criteria and was developed by Japanese statistician Hirotugo Akaike, and BIC, which stands for Bayesian Information Criteria, are also used as criteria for model robustness. The goal is to minimize these values to get a better model.
- **Omnibus and Prob(Omnibus):** Omnibus test checks the normality of the residuals once the model is deployed. If the value is zero, it means the residuals are perfectly normal. For a model to be robust, besides checking $R$-squared and other rubrics, the residual distribution is also required to be normal. In other words, the residual should not follow any pattern when plotted against the fitted values.
- **Skew and Kurtosis:** Skew values tells us the skewness of the residual distribution. Normally distributed variables have $0$ skew value, right-skewed distributions have positive skew values and lefth-skewed distributions have negative skew values. Kurtosis is a measure of light-tailed or heavy-tailed distribution compared to the normal distribution. High kurtosis indicates the distribution is too narrow and low kurtosis indicates the distribution is too flat. A kurtosis value between $-2$ and $+2$ is good to prove normalcy.
- **Durbin-Watson:** The Durbin-Watson statistic provides a measure of autocorrelation in the residuals. If the residual values are autocorrelated, the model becomes biased. This simply means that one value should not be depending on any of the previous values. An ideal value for this test ranges from $0$ to $4$.
- **Jarque-Bera (JB) and Prob(JB):** Jarque-Bera (JB) and Prob(JB) are similar to Omni test measuring the normalcy of the residuals.
- **Condition Number:** High condition number indicates that possibly there is multicollinearity present in the dataset. If only one variable is used as predictor, this value is low and can be ignored. We can proceed like stepwise regression and see if there is any multicollinearity added when additional variables are included.

In [ ]:
# Train-test splitting
# ==============================================================================
X = data[['at_bat']]
y = data['runs']

X_train, X_test, y_train, y_test = train_test_split(
                                        X.values.reshape(-1,1),
                                        y.values.reshape(-1,1),
                                        train_size   = 0.8,
                                        random_state = 1234,
                                        shuffle      = True
                                    )

In [ ]:
data_train = pd.DataFrame(np.hstack((X_train, y_train)), columns=['at_bat', 'runs'])
data_test = pd.DataFrame(np.hstack((X_test, y_test)), columns=['at_bat', 'runs'])

#### Model fitting with intercept

In [ ]:
# Model fitting using a formula
# ==============================================================================
modelf = smf.ols(formula = 'runs ~ at_bat', data = data_train)
modelf = modelf.fit()
modelf.summary()

In [ ]:
# Model fitting using matrices
# ==============================================================================
# Add a column of 1s to the predictor matrix
X_train_int = sm.add_constant(X_train, prepend=True)
modelm = sm.OLS(endog=y_train, exog=X_train_int,)
modelm = modelm.fit()
modelm.summary()

#### Model fitting without intercept

In [ ]:
# Model fitting using a formula
# ==============================================================================

modelf_noint = smf.ols(formula = 'runs ~ at_bat + 0', data = data_train)
modelf_noint = modelf_noint.fit()
modelf_noint.summary()

In [ ]:
# Model fitting using matrices
# ==============================================================================
# Add a column of 1s to the predictor matrix
modelm_noint = sm.OLS(endog=y_train, exog=X_train,)
modelm_noint = modelm_noint.fit()
modelm_noint.summary()

## Confidence intervals for the coefficients

In [ ]:
# Confidence intervals for the coefficients
# ==============================================================================
modelf.conf_int(alpha=0.05)

## Residual analysis

In [ ]:
# Residual analysis
# ==============================================================================
y_train = y_train.flatten()
prediction_train = modelf.predict(exog = data_train)
residues_train   = prediction_train - y_train

In [ ]:
residues_train.sum()

In [ ]:
residues_train.max()

### Graphs

In [ ]:
#plt.rcParams['image.cmap'] = "fall"

In [ ]:
# Graphs
# ==============================================================================
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(9, 8))

# axes[0,0]
sns.scatterplot(
    x          = y_train,
    y          = prediction_train,
    edgecolors =(0, 0, 0),
    alpha      = 0.4,
    ax         = axes[0, 0]
)

points = [(y_train.min(),prediction_train.min()), (y_train.max(), prediction_train.max())]  # x, y coordinates
line_data = pd.DataFrame(points, columns=['x', 'y'])
sns.lineplot(
    x          = 'x',
    y          = 'y',
    data       = line_data, 
    linestyle  = '--', 
    color      = 'gray', 
    lw         = 2,
    ax         = axes[0, 0]
)

axes[0, 0].set_title('Predicted value vs real value', fontsize = 10, fontweight = "bold")
axes[0, 0].set_xlabel('Real')
axes[0, 0].set_ylabel('Prediction')
axes[0, 0].tick_params(labelsize = 7)

# axes[0,1]
sns.scatterplot(
    x          = list(range(len(y_train))),
    y          = residues_train,
    edgecolors = (0, 0, 0),
    alpha      = 0.4,
    ax         = axes[0, 1]
)

axes[0, 1].axhline(y = 0, linestyle = '--', color = 'gray', lw=2)
axes[0, 1].set_title('Residues', fontsize = 10, fontweight = "bold")
axes[0, 1].set_xlabel('id')
axes[0, 1].set_ylabel('Residue')
axes[0, 1].tick_params(labelsize = 7)

# axes[1, 0]
sns.histplot(
    data     = residues_train,
    stat     = "density",
    kde      = True,
    line_kws = {'linewidth': 1},
    alpha    = 0.3,
    ax       = axes[1, 0]
)

axes[1, 0].set_title('Distribution of residues', fontsize = 10, fontweight = "bold")
axes[1, 0].set_xlabel("Residue")
axes[1, 0].tick_params(labelsize = 7)

# axes[1, 1]
sm.qqplot(
    residues_train,
    fit   = True,
    line  = 'q',
    ax    = axes[1, 1], 
    alpha = 0.4,
    lw    = 2
)

axes[1, 1].set_title('Q-Q residues', fontsize = 10, fontweight = "bold")
axes[1, 1].tick_params(labelsize = 7)

# axes[2, 0]
sns.scatterplot(
    x          = prediction_train,
    y          = residues_train,
    edgecolors = (0, 0, 0),
    alpha      = 0.4,
    ax         = axes[2, 0]
)

axes[2, 0].axhline(y = 0, linestyle = '--', color = 'grey', lw=2)
axes[2, 0].set_title('Residues vs prediction', fontsize = 10, fontweight = "bold")
axes[2, 0].set_xlabel('Prediction')
axes[2, 0].set_ylabel('Residue')
axes[2, 0].tick_params(labelsize = 7)

# Eliminate empty axes
fig.delaxes(axes[2,1])

fig.tight_layout()
plt.subplots_adjust(top=0.9)
fig.suptitle('Residual analysis', fontsize = 12, fontweight = "bold");

### Normality test

We now verify if the residues follow a normal distribution using to statistical tests: *Shapiro-Wilk test* and *D'Agostino's K-squared test*. The latter includes a summary from **Statsmodels** with the name *Omnibus*.

On both tests, the null hypothesis considers that the data follows a normal distribution. Hence, if the *$p$-value* is not smaller than the selected significance level *alpha* there is no evidence to reject the statement that the data is normally distributed.

In [ ]:
# Shapiro-Wilk test
# ==============================================================================
shapiro_test = stats.shapiro(residues_train)
shapiro_test

In [ ]:
# D'Agostino's K-squared test
# ==============================================================================
k2, p_value = stats.normaltest(residues_train)#prediction_train = modelf.predict(exog = X_train)

print(f"Statistic= {k2}, p-value = {p_value}")

## Predictions

Now that we have trained our model, we can obtain predictions for new data. Models from **statsmodels** allow us to calculate predictions in two ways:

- `.predict()`: only gives predicted values.

- `.get_prediction().summary_frame()`: gives confidence intervals in addition to predicted values.

In [ ]:
# Predictions with 95% confidence intervals
# ==============================================================================
predictions = modelf.get_prediction(exog = data_train).summary_frame(alpha=0.05)
predictions.head(4)

## Graphical representation

Now, we plot the least-squares regression line, as well as upper and lower limits for confidence intervals. With this graph it is possible to identify a region for the *mean* value of the response variable.

In [ ]:
# Predictions with 95% confidence intervals
# ==============================================================================
predictions = modelf.get_prediction(exog = data_train).summary_frame(alpha=0.05)
predictions['x'] = X_train[:, 0]
predictions['y'] = y_train
predictions = predictions.sort_values('x')

# Graph
# ==============================================================================
sns.regplot(
    x     = 'x',
    y     ='y',
    data  = predictions,
).set_title('Linear regression fit');

## Model error test

In [ ]:
# Model error test
# ==============================================================================
#X_test = sm.add_constant(X_test, prepend=False)
predictions = modelf.predict(exog = data_test)
rmse = root_mean_squared_error(
        y_true  = y_test,
        y_pred  = predictions
       )
print("")
print(f"The test error (rmse) is {rmse}, which is {rmse/mean_predicted_variable*100:.2f}% of the mean of `runs`.")

## Interpreting the results

The `coef` column gives the estimated value for both of the parameters of the linear model ($\hat{a}$ and $\hat{b}$). These correspond to the intercept and the slope of the line. Standard errors, $t$-statistic value and $p$-value for each of the parameters is also shown. This allows us to determine if the predictors are significantly different than 0, that is, they are important for the model. We see that such *$p$-values* are smaller than 0.05.

`R-squared` indicates that the model is able to explain 27.1% of the observerd variability for the response variable. Furthermore, the *$p$-value* obtained in the F-test (`Prob (F-statistic) = 0.00906`) indicates that there is sufficient evidence that the variance explained by the model is greater than that of random chance.

So, the linear model thus generated follows the equation:

$$runs = -2367.7028 + 0.5529 at\_bat$$
 
For each increment in At-bats, the number of runs increases by 0.5529 **on average**

The model error test is 59.34, so the model predictions get farther from the real value by 59.34 **on average**. This error is less than 10% the average of `runs`, so we can conclude the regression is valid.

## References

Regresión lineal con Python by Joaquín Amat Rodrigo, available under a Attribution 4.0 International (CC BY 4.0) at https://www.cienciadedatos.net/documentos/py10-regresion-lineal-python.html